# PREPARE ENVIRONMENT

In [1]:
import os
import sys
import yaml
from pathlib import Path

# Add the parent directory (where "modules" is located) to the Python path
notebook_dir = os.getcwd()
parent_dir = os.path.dirname(notebook_dir)
sys.path.append(parent_dir)

# import functions from modules
from modules import llm_database_iteration_functions as ldif
from modules import llm_iteration_functions as lif

# get config data
conf = yaml.safe_load(Path(os.path.join(parent_dir, "config.yaml")).read_text(encoding="utf-8"))

# MODEL PULLING

Pull or download the `llama3.2:1b` model from the Ollama library.

In [2]:
import ollama

# download model to iterate with
ollama.pull("llama3.2:1b")

ProgressResponse(status='success', completed=None, total=None, digest=None)

# SQL - RAG SIMULATION 1

After displaying a welcome message and showing possible options, the user's response is simulated to trigger specific actions. These actions are linked to SQL queries created using `write_query` and executed through the `QueryExecution process`. The system then returns the desired results based on the simulated user choices.

In [3]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.llms import Ollama

# STEP1: Load Ollama model
llm = Ollama(model="llama3.2:1b", temperature = 0) # always same result for same input

# STEP2: Get database_path
data_folder = os.path.join(parent_dir, conf["DATA_PATH"])
database_path = os.path.join(data_folder, conf["DATABASE_NAME"])

##########################
# SIMULATE CONVERSATION
##########################

# GIVE WELCOME MESSAGE AND PROMPT FIRST OPTION MENU
response = llm.invoke(lif.get_welcome_prompt().format())
print("<MODEL>:", lif.translate_text_with_Elia(response, "en", "eu"))

####################################################
# SIMULATE USER ANSWER:
#     1: Know the information that is available
####################################################
# simulate answer = 1, visualize next options
answer_sim = 1
user_answer = lif.get_welcome_menu_user_answer(answer_sim)
print("---\n<USER>:", user_answer)

# Create chain to invoke aswer1 submenu prompt to visualize
chain1 = LLMChain(llm=llm, prompt=lif.get_submenu1_prompt())
response1 = llm.invoke(lif.get_submenu1_prompt().template)
print("---\n<MODEL>:", lif.translate_text_with_Elia(response1, "en", "eu"))

####################################################
# SIMULATE USER ANSWER:
#     1: Consult the information tables
####################################################
# get database_path
data_folder = os.path.join(parent_dir, conf["DATA_PATH"])
database_path = os.path.join(data_folder, conf["DATABASE_NAME"])

# Simulate distinct answers one after other
simulated_answers = [1, 2, 3, 4, 5, 6]
dest_lang = "eu"

for answer_sim in simulated_answers:
    user_answer = lif.get_submenu1_answers(answer_sim)
    print("---\n<USER>:", user_answer)

    # create action dictionary
    actions = {
        1: lambda: ldif.get_database_tables(database_path, conf, True, True, dest_lang),
        2: lambda: ldif.get_rows_number_all_tables(database_path, llm, conf, dest_lang),
        3: lambda: ldif.get_recipe_count_per_category(database_path, llm,["dish_order", "cook_technique", "localization"], dest_lang),
        4: lambda: ldif.get_ingredient_category_top3(database_path, False, dest_lang), # Top 3 ingredient categories with the most recipes
        5: lambda: ldif.get_ingredient_category_top3(database_path, True, dest_lang), # Top 3 ingredient categories with the fewest recipes
        6: lambda: ldif.get_most_used_ingredient_per_category(database_path, dest_lang)
    }

    # Execute corresponding action according to simulation answer
    action = actions.get(answer_sim)
    if action:
        action()
    else:        
        # prompt error message in basque
        print(lif.translate_text_with_Elia(f"<ERROR>: There is no action for {answer_sim}", "en", dest_lang))

<MODEL>: Kaixo!
Sukaldaritzan espezializatutako laguntzaile birtuala naiz.
Errezetak bilatzen eta zure lehentasunetan oinarritutako menuak sortzen laguntzera nator.
Bi aukera dituzu:
1. Kontsultatu informazioa: Kontsultatu kategoria, osagai, erabilera eta ezaugarri bakoitzaren errezeta-taulak eta kantitateak.
2. Menu edo plater bat osatzea: Menuko aukerak aztertzea, osagai edo plateren agindu zehatzak erabiliz.
Zein aukera hautatu nahi duzu?
---
<USER>: Ze ongi, 1 aukera aukeratzen dut, zein aukera eskaintzen dituzu?


/tmp/ipykernel_962/1188583212.py:30: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  chain1 = LLMChain(llm=llm, prompt=lif.get_submenu1_prompt())


---
<MODEL>: Hona hemen aukerak:
1. Kontsultatu informazio-taulak.
2. Kontsultatu informazio-taulen errenkaden kopurua.
3. Kontsultatu Taula kategoriako errezeta kopurua
4. Kontsultatu errezeta gehien dituzten osagaien kategoriak (top3).
5. Kontsultatu errezeta gutxien dituzten osagaien kategoriak (top3).
6. Kontsultatu osagaien kategorian gehien erabiltzen den osagaia.
Zein aukera hautatu nahiko zenuke?
---
<USER>: 1 aukera aukeratzen dut, emaidazu informazio-taula zerrenda.
---
<MODEL>: Taula hauek daude datu-basean:
	- 'cook_technique': Sukaldatze teknikak: erregosi, labekatu...
	- 'dish_order': Platera ordena: lehen platera, bigarren platera...
	- 'ingredient': Errezetetan erabiltzen diren banakako osagaien zerrenda.
	- 'ingredient_category': Osagaien kategoriak, hala nola barazkiak, haragia, etab.
	- 'localization': Errezeten kokapenari buruzko informazioa.
	- 'recipe': Errezetak eta haien xehetasunak, osagaiak eta prestaketa-urratsak biltzen dituen taula.
---
<USER>: 2 aukera auk